## Project objective

The project asks us to use exploratory data analysis to generate insights for a business stakeholder who is considering starting a movie studio.

The uploaded dataset is a TMDB movie dataset. It contains movie titles, genres represented by IDs, release dates, popularity, average ratings, and vote counts.

**Important limitation:** this dataset does not contain box-office revenue. Therefore, the analysis below uses popularity, ratings, vote counts, release patterns, and genre-count information as audience-interest indicators. It should not be described as a direct analysis of box-office revenue.

## 1. Import the libraries

We use pandas for data preparation and analysis, matplotlib and seaborn for visualization.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


## 2. Load the data

The source file is compressed, so pandas can read it directly without manually extracting it.


In [2]:
file_path = r"zippedData/tmdb.movies.csv.gz"
movies = pd.read_csv(file_path, compression="gzip")

movies.head()


,Unnamed: 0,genre_ids,id,original_language,original_title,popularity,release_date,title,vote_average,vote_count
0,0,"[12, 14, 10751]",12444,en,Harry Potter and the Deathly Hallows: Part 1,33.533,2010-11-19,Harry Potter and the Deathly Hallows: Part 1,7.7,10788
1,1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,2010-03-26,How to Train Your Dragon,7.7,7610
2,2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,2010-05-07,Iron Man 2,6.8,12368
3,3,"[16, 35, 10751]",862,en,Toy Story,28.005,1995-11-22,Toy Story,7.9,10174
4,4,"[28, 878, 12]",27205,en,Inception,27.920,2010-07-16,Inception,8.3,22186


## 3. Understand the dataset

Before cleaning, inspect the number of rows, columns, data types, missing values, and duplicate records.


In [ ]:
print("Rows and columns:", movies.shape)
print("\nColumn names:")
print(movies.columns.tolist())

print("\nData types:")
print(movies.dtypes)

print("\nMissing values:")
print(movies.isna().sum())

print("\nDuplicate rows:", movies.duplicated().sum())
print("Duplicate movie IDs:", movies["id"].duplicated().sum())


Rows and columns: (26517, 10)

Column names:
['Unnamed: 0', 'genre_ids', 'id', 'original_language', 'original_title', 'popularity', 'release_date', 'title', 'vote_average', 'vote_count']

Data types:
Unnamed: 0             int64
genre_ids             object
id                     int64
original_language     object
original_title        object
popularity           float64
release_date          object
title                 object
vote_average         float64
vote_count             int64
dtype: object

Missing values:
Unnamed: 0           0
genre_ids            0
id                   0
original_language    0
original_title       0
popularity           0
release_date         0
title                0
vote_average         0
vote_count           0
dtype: int64

Duplicate rows: 0
Duplicate movie IDs: 1020


## 4. Data cleaning

The cleaning process will:

1. Remove the unnecessary `Unnamed: 0` column.
2. Remove duplicate records.
3. Standardize column names.
4. Convert `release_date` into a datetime column.
5. Convert numeric fields to numeric types.
6. Remove rows missing the core movie ID or release date.
7. Check that ratings are between 0 and 10.
8. Remove negative popularity and vote counts.
9. Make movie IDs unique.
10. Create `release_year` and `genre_count` columns for analysis.


In [ ]:
# Remove the unnecessary index column
movies = movies.drop(columns=["Unnamed: 0"], errors="ignore")

# Remove exact duplicate records
movies = movies.drop_duplicates()

# Standardize column names
movies.columns = (
    movies.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Convert release date to datetimec
movies["release_date"] = pd.to_datetime(
    movies["release_date"], errors="coerce"
)

# Convert numeric columns
numeric_columns = ["id", "popularity", "vote_average", "vote_count"]

for column in numeric_columns:
    movies[column] = pd.to_numeric(movies[column], errors="coerce")

# Remove rows missing essential fields
movies = movies.dropna(subset=["id", "release_date"])

# Keep valid numeric values
movies = movies[movies["vote_average"].between(0, 10)]
movies = movies[movies["vote_count"] >= 0]
movies = movies[movies["popularity"] >= 0]

# Make movie IDs unique
movies = movies.drop_duplicates(subset=["id"], keep="first")

# Create useful analysis columns
movies["release_year"] = movies["release_date"].dt.year
movies["genre_count"] = movies["genre_ids"].str.count(",") + 1

movies.head()


,genre_ids,id,original_language,original_title,popularity,release_date,title,vote_average,vote_count,release_year,genre_count
0,"[12, 14, 10751]",12444,en,Harry Potter and the Deathly Hallows: Part 1,33.533,2010-11-19,Harry Potter and the Deathly Hallows: Part 1,7.7,10788,2010,3
1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,2010-03-26,How to Train Your Dragon,7.7,7610,2010,4
2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,2010-05-07,Iron Man 2,6.8,12368,2010,3
3,"[16, 35, 10751]",862,en,Toy Story,28.005,1995-11-22,Toy Story,7.9,10174,1995,3
4,"[28, 878, 12]",27205,en,Inception,27.920,2010-07-16,Inception,8.3,22186,2010,3


## 5. Validate the cleaned data

Validation confirms that the cleaning operations produced a usable dataset.


In [ ]:
print("Cleaned shape:", movies.shape)

print("\nMissing values after cleaning:")
print(movies.isna().sum())

print("\nDuplicate rows after cleaning:", movies.duplicated().sum())
print("Duplicate movie IDs after cleaning:", movies["id"].duplicated().sum())

print("\nRelease date range:")
print(movies["release_date"].min(), "to", movies["release_date"].max())

print("\nRating range:")
print(movies["vote_average"].min(), "to", movies["vote_average"].max())


Cleaned shape: (25497, 11)

Missing values after cleaning:
genre_ids            0
id                   0
original_language    0
original_title       0
popularity           0
release_date         0
title                0
vote_average         0
vote_count           0
release_year         0
genre_count          0
dtype: int64

Duplicate rows after cleaning: 0
Duplicate movie IDs after cleaning: 0

Release date range:
1930-04-29 00:00:00 to 2020-12-25 00:00:00

Rating range:
0.0 to 10.0
